A. Extraer la información del archivo.

In [2]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')
# Cargar el archivo CSV, especificando el delimitador como coma, la fila de encabezado y el motor de python
df = pd.read_csv('/content/drive/MyDrive/Certificación Ciencia de Datos/Reto semana 5/country_vaccinations.csv', sep=',', header=0, engine='python')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


B. Mostrar la estructura y tipos de datos de cada columna para identificar qué operaciones puedes realizar con cada una de ellas, asegurándote que las columnas con fechas sean del tipo datetime64.

In [3]:
# Mostrar las columnas disponibles en el DataFrame
print("Columnas disponibles en el DataFrame:")
print(df.columns)

# Mostrar las primeras 5 filas del DataFrame para inspeccionar los datos y la estructura
print("\nPrimeras 5 filas del DataFrame:")
print(df.head())
# Convertir la columna 'date' a datetime64
df['date'] = pd.to_datetime(df['date'])

# Mostrar la estructura y tipos de datos
df.info()

Columnas disponibles en el DataFrame:
Index(['country_vaccinations'], dtype='object')

Primeras 5 filas del DataFrame:
                                                                                                                                                                                                                                                                                                                                                         country_vaccinations
country     iso_code date       total_vaccinations people_vaccinated people_fully_vaccinated daily_vaccinations_raw daily_vaccinations total_vaccinations_per_hundred people_vaccinated_per_hundred people_fully_vaccinated_per_hundred daily_vaccinations_per_million vaccines                                           source_name                          source_website
Afghanistan AFG      2021-02-22 0.0                0.0               NaN                     NaN                    NaN                0.0           

KeyError: 'date'

Comprobamos si el DataFrame se cargó como una sola columna que contiene la línea completa como cadena.
Esto ocurre cuando el separador predeterminado (coma) no se identifica correctamente
o el archivo tiene alguna estructura inusual que hace que pandas no lo divida.
La salida indica que la primera columna se llama 'country_vaccinations' y contiene la línea completa.

In [4]:
try:
    if len(df.columns) == 1 and df.columns[0] == 'country_vaccinations' and df.iloc[0, 0].count(',') > 0:
        print("\n--- Advertencia: El archivo CSV parece no haberse cargado correctamente. ---")
        print("El archivo se cargó como una sola columna donde cada celda contiene la línea completa.")
        print("Se intentará parsear la columna única usando la coma como delimitador.")

        # La primera entrada en la columna única es la línea de encabezado real
        header_line = df.iloc[0, 0]
        column_names = header_line.split(',')

        # Aplicar la división al resto de las filas en esa columna única
        # y crear un nuevo DataFrame a partir de ella
        data_rows = df.iloc[1:]['country_vaccinations'].str.split(',', expand=True)
        data_rows.columns = column_names
        df = data_rows

        print("\n--- Archivo CSV parseado exitosamente desde una columna única ---")

    # Si la condición anterior no se cumplió, pero el resultado es un MultiIndex
    elif isinstance(df.index, pd.MultiIndex):
        print("\n--- Advertencia: El archivo CSV parece no haberse cargado correctamente. ---")
        print("El archivo se cargó con el encabezado mezclado en el MultiIndex.")
        print("Intentando reconstruir el DataFrame correctamente desde el MultiIndex.")

        # Convertir el MultiIndex a un DataFrame, lo que convierte sus niveles en columnas
        # En este punto, las columnas del nuevo DataFrame se llamarán 'None', 'None', etc.
        df_reconstructed = df.index.to_frame(index=False)

        # Establecer los nombres de las columnas a partir de la primera fila
        df_reconstructed.columns = df_reconstructed.iloc[0]

        # Eliminar la fila que ahora es el encabezado y usar el resto como datos
        df = df_reconstructed.iloc[1:].copy()

        print("\n--- Archivo CSV reconstruido exitosamente desde el MultiIndex y encabezado corregido ---")

    print("\n--- Archivo CSV cargado exitosamente ---")

except FileNotFoundError:
    # Assuming file_path was intended to be defined elsewhere; replacing with a generic message.
    print(f"¡Error! No se encontró el archivo. Asegúrate de que el nombre del archivo es correcto y que ha sido subido.")
    print("Por favor, sube tu archivo CSV o proporciona la ruta correcta.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo: {e}")


--- Advertencia: El archivo CSV parece no haberse cargado correctamente. ---
El archivo se cargó con el encabezado mezclado en el MultiIndex.
Intentando reconstruir el DataFrame correctamente desde el MultiIndex.

--- Archivo CSV reconstruido exitosamente desde el MultiIndex y encabezado corregido ---

--- Archivo CSV cargado exitosamente ---


Ahora si, intentamos mostrar la información del dataset en el formato que se espera y se hace la modificación de la columna date al formato datetime64

In [5]:
# Mostrar las columnas disponibles en el DataFrame
print("Columnas disponibles en el DataFrame:")
print(df.columns)

# Mostrar las primeras 5 filas del DataFrame para inspeccionar los datos y la estructura
print("\nPrimeras 5 filas del DataFrame:")
print(df.head())
# Convertir la columna 'date' a datetime64
df['date'] = pd.to_datetime(df['date'])

# Mostrar la estructura y tipos de datos
df.info()

Columnas disponibles en el DataFrame:
Index(['country', 'iso_code', 'date', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated',
       'daily_vaccinations_raw', 'daily_vaccinations',
       'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred',
       'people_fully_vaccinated_per_hundred', 'daily_vaccinations_per_million',
       'vaccines', 'source_name'],
      dtype='object', name=0)

Primeras 5 filas del DataFrame:
0      country iso_code        date total_vaccinations people_vaccinated  \
1  Afghanistan      AFG  2021-02-22                0.0               0.0   
2  Afghanistan      AFG  2021-02-23                NaN               NaN   
3  Afghanistan      AFG  2021-02-24                NaN               NaN   
4  Afghanistan      AFG  2021-02-25                NaN               NaN   
5  Afghanistan      AFG  2021-02-26                NaN               NaN   

0 people_fully_vaccinated daily_vaccinations_raw daily_vaccinations  \
1           

In [6]:
#Todos los datos están como tipo object y algunos deberían de convertirse a tipo de datos numéricos para poder realizar operaciones matemáticas con ellos, procedamos a ello:
# Convertir columnas numéricas a float
# Identificar columnas que deberían ser numéricas (excluyendo 'country', 'iso_code', 'vaccines', 'source_name')
numeric_cols = [
    'total_vaccinations',
    'people_vaccinated',
    'people_fully_vaccinated',
    'daily_vaccinations_raw',
    'daily_vaccinations',
    'total_vaccinations_per_hundred',
    'people_vaccinated_per_hundred',
    'people_fully_vaccinated_per_hundred',
    'daily_vaccinations_per_million'
]

for col in numeric_cols:
    # Usar errors='coerce' para convertir valores no numéricos a NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(f"Columnas numéricas convertidas a tipo float: {numeric_cols}")

print("\n--- Fin de la sección de limpieza de datos ---")
print("\n--- Primeras 5 filas DESPUÉS de limpieza ---")
print(df.head())
print("\n--- Información general del DataFrame DESPUÉS de limpieza ---")
print(df.info())


Columnas numéricas convertidas a tipo float: ['total_vaccinations', 'people_vaccinated', 'people_fully_vaccinated', 'daily_vaccinations_raw', 'daily_vaccinations', 'total_vaccinations_per_hundred', 'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred', 'daily_vaccinations_per_million']

--- Fin de la sección de limpieza de datos ---

--- Primeras 5 filas DESPUÉS de limpieza ---
0      country iso_code       date  total_vaccinations  people_vaccinated  \
1  Afghanistan      AFG 2021-02-22                 0.0                0.0   
2  Afghanistan      AFG 2021-02-23                 NaN                NaN   
3  Afghanistan      AFG 2021-02-24                 NaN                NaN   
4  Afghanistan      AFG 2021-02-25                 NaN                NaN   
5  Afghanistan      AFG 2021-02-26                 NaN                NaN   

0  people_fully_vaccinated  daily_vaccinations_raw  daily_vaccinations  \
1                      NaN                     NaN               

C. Determinar la cantidad de vacunas aplicadas de cada compañía (con base en cómo lo reporta cada país en la columna vaccines, en otras palabras, agrupe por vaccines y realice la sumatoria).

In [7]:
# Agrupar por 'vaccines' y sumamos las vacunas diarias ('daily_vaccinations')
vacunas_por_compania = df.groupby('vaccines')['daily_vaccinations'].sum()
print("Cantidad de vacunas por compañía:")
print(vacunas_por_compania)

Cantidad de vacunas por compañía:
vaccines
Abdala, Johnson&Johnson, Oxford/AstraZeneca, Pfizer/BioNTech, Soberana02, Sputnik Light, Sputnik V                              9616160.0
Abdala, Moderna, Oxford/AstraZeneca, Pfizer/BioNTech, Sinopharm/Beijing, Sputnik V                                            201816053.0
Abdala, Sinopharm/Beijing, Sinovac, Soberana02, Sputnik Light, Sputnik V                                                       37861146.0
Abdala, Soberana Plus, Soberana02                                                                                              33802957.0
COVIran Barekat, Covaxin, FAKHRAVAC, Oxford/AstraZeneca, Razi Cov Pars, Sinopharm/Beijing, Soberana02, SpikoGen, Sputnik V    146357015.0
                                                                                                                                 ...     
Pfizer/BioNTech, Sinovac, Turkovac                                                                                            147

D. Obtener la cantidad de vacunas aplicadas en todo el mundo

In [8]:
# Sumatoria global de las vacunas diarias aplicadas
total_mundo = df['daily_vaccinations'].sum()
print(f"Total de vacunas aplicadas en el mundo: {total_mundo}")

Total de vacunas aplicadas en el mundo: 11320239871.0


E. Calcular el promedio de vacunas aplicadas por país.

In [9]:
# Agrupar por país y sacar el promedio de la columna de vacunas diarias
promedio_pais = df.groupby('country')['daily_vaccinations'].mean()
print("Promedio de vacunas aplicadas por país:")
print(promedio_pais)

Promedio de vacunas aplicadas por país:
country
Afghanistan          14610.681934
Albania               6276.210046
Algeria              33904.356436
Andorra                367.716019
Angola               44821.457584
                         ...     
Wales                15518.411765
Wallis and Futuna       33.886486
Yemen                 2556.115756
Zambia                9649.805158
Zimbabwe             21669.066832
Name: daily_vaccinations, Length: 223, dtype: float64


F. Determinar la cantidad de vacunas aplicadas el día 29/01/21 en todo el mundo.

In [10]:
# Se filtra por la fecha específica y se suman las vacunas de ese día
vacunas_29_01 = df[df['date'] == '2021-01-29']['daily_vaccinations'].sum()
print(f"Vacunas aplicadas el 29/01/21 en todo el mundo: {vacunas_29_01}")

Vacunas aplicadas el 29/01/21 en todo el mundo: 4884052.0


G. Crear un dataframe nuevo llamado conDiferencias que contenga los datos originales y una columna derivada (diferencias) con las diferencias de aplicación entre las columnas daily_vaccionations y daily_vaccionations_raw.

In [11]:
conDiferencias = df.copy()
# Se saca la diferencia matemática entre ambas columnas
conDiferencias['diferencias'] = conDiferencias['daily_vaccinations'] - conDiferencias['daily_vaccinations_raw']
conDiferencias[['daily_vaccinations', 'daily_vaccinations_raw', 'diferencias']].head()

,daily_vaccinations,daily_vaccinations_raw,diferencias
1,NaN,NaN,NaN
2,1367.0,NaN,NaN
3,1367.0,NaN,NaN
4,1367.0,NaN,NaN
5,1367.0,NaN,NaN


H. Obtener el periodo de tiempo entre el registro con fecha más reciente y el registro con fecha más antigua.

In [12]:
periodo_tiempo = df['date'].max() - df['date'].min()
print(f"El periodo de tiempo registrado es de: {periodo_tiempo}")

El periodo de tiempo registrado es de: 482 days 00:00:00


I. Crear un dataframe nuevo denominado conCantidad que contenga los datos originales y una columna derivada (canVac) con la cantidad de vacunas utilizadas cada día (usar la columna vaccines y separar por el carácter ,).

In [13]:
conCantidad = df.copy()
# Se transforma a tipo string, separamos por coma y contamos los elementos resultantes de la lista
conCantidad['canVac'] = conCantidad['vaccines'].apply(lambda x: len(str(x).split(',')))
conCantidad[['vaccines', 'canVac']].head()

,vaccines,canVac
1,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",4
2,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",4
3,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",4
4,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",4
5,"Johnson&Johnson, Oxford/AstraZeneca, Pfizer/Bi...",4


J. Generar un dataframe denominado antes20 con todos los registros que se hayan realizado antes del 20 de diciembre de 2020.

In [16]:
antes20 = df[df['date'] < '2020-12-20']
print(f"Registros antes del 20 de diciembre de 2020: {len(antes20)}")
print (antes20.head)

Registros antes del 20 de diciembre de 2020: 70
<bound method NDFrame.head of 0            country iso_code       date  total_vaccinations  \
13404         Canada      CAN 2020-12-14                 5.0   
13405         Canada      CAN 2020-12-15               727.0   
13406         Canada      CAN 2020-12-16              3025.0   
13407         Canada      CAN 2020-12-17              7279.0   
13408         Canada      CAN 2020-12-18             11296.0   
...              ...      ...        ...                 ...   
82363  United States      USA 2020-12-15             84638.0   
82364  United States      USA 2020-12-16            244549.0   
82365  United States      USA 2020-12-17            517161.0   
82366  United States      USA 2020-12-18            933551.0   
82367  United States      USA 2020-12-19           1115437.0   

0      people_vaccinated  people_fully_vaccinated  daily_vaccinations_raw  \
13404                5.0                      NaN                     NaN   

K. Obtener un dataframe denominado pfizer con todos los registros donde se haya utilizado la vacuna Pfizer.

In [18]:
# Se filtra buscando la cadena 'Pfizer' dentro de la columna vaccines, ignorando valores nulos
pfizer = df[df['vaccines'].str.contains('Pfizer', case=False, na=False)]
print(f"Registros donde se utilizó Pfizer: {len(pfizer)}")
print (pfizer.head)

Registros donde se utilizó Pfizer: 64193
<bound method NDFrame.head of 0          country  iso_code       date  total_vaccinations  \
1      Afghanistan       AFG 2021-02-22                 0.0   
2      Afghanistan       AFG 2021-02-23                 NaN   
3      Afghanistan       AFG 2021-02-24                 NaN   
4      Afghanistan       AFG 2021-02-25                 NaN   
5      Afghanistan       AFG 2021-02-26                 NaN   
...            ...       ...        ...                 ...   
85070        Wales  OWID_WLS 2022-03-24           6921195.0   
85071        Wales  OWID_WLS 2022-03-25           6923298.0   
85072        Wales  OWID_WLS 2022-03-26           6923706.0   
85073        Wales  OWID_WLS 2022-03-27           6925183.0   
85074        Wales  OWID_WLS 2022-03-28           6927437.0   

0      people_vaccinated  people_fully_vaccinated  daily_vaccinations_raw  \
1                    0.0                      NaN                     NaN   
2                 

I. Almacenar los dataframes generados (conDiferencias, conCantidad, antes20 y pfizer) en un archivo de Excel denominado resultadosReto.xlsx, donde cada dataframe ocupe una hoja diferente.

In [19]:
# Usamos pd.ExcelWriter para guardar múltiples hojas en un solo archivo
with pd.ExcelWriter('resultadosReto.xlsx') as writer:
    conDiferencias.to_excel(writer, sheet_name='conDiferencias', index=False)
    conCantidad.to_excel(writer, sheet_name='conCantidad', index=False)
    antes20.to_excel(writer, sheet_name='antes20', index=False)
    pfizer.to_excel(writer, sheet_name='pfizer', index=False)

print("Archivo 'resultadosReto.xlsx' generado y guardado exitosamente.")

Archivo 'resultadosReto.xlsx' generado y guardado exitosamente.
